# FINRL Walk-Forward Experiment

This notebook runs the direct-feature walk-forward experiment runner and visualizes portfolio performance against the S&P 500 / SPY benchmark.

Use small synthetic data locally. Use Colab for the full configured universe, direct portfolio optimization, and full walk-forward experiments.

In [ ]:
%cd /content
![ -d FINRL ] || git clone https://github.com/nidarshans/FINRL.git
%cd /content/FINRL

In [ ]:
%cd /content/FINRL
%pip install -e .


In [1]:
# Keep notebook imports pointed at the live workspace package.

from datetime import date, timedelta

import polars as pl

from finrl.backtest.walk_forward import WalkForwardConfig
from finrl.dpo_jax import DPOConfig
from finrl.data import (
    MarketDataBundle,
    MarketDataConfig,
    UniverseConfig,
    build_rebalance_calendar,
    compute_open_to_open_returns,
    download_ohlcv,
)
from finrl.data.download import download_macro_series
from finrl.env.trading_env import EnvConfig
from finrl.experiments import (
    ExperimentConfig,
    RawExperimentData,
    build_allocation_figure,
    build_drawdown_figure,
    build_holdings_heatmap_granular,
    build_performance_figure,
    build_regime_portfolio_figure,
    build_spectral_figure,
    metrics_to_frame,
    run_walk_forward_experiment,
)
from finrl.features import FeatureConfig, build_feature_bundle, selected_direct_allocation_indices
from finrl.features.preprocessing import PreprocessingConfig
from finrl.features.schema import FeatureBundle
import numpy as np

## Prepared Data Contract

The runner expects prepared feature and return tables:

- `FeatureBundle` with asset and macro features plus a dummy 20-column spectral compatibility table.
- `returns`: Polars DataFrame with `decision_date` and one return column per tradable asset.
- `spy_returns`: Polars DataFrame with `decision_date` and `spy_return` for the same holding periods.

Replace the synthetic fixture below with the output of the data, feature, preprocessing, and return-preparation pipeline for full experiments. DPO trains the direct allocation head over explicitly routed asset features.

## Run With Real yfinance Data

Edit `TICKERS`, `START`, `END`, and `MAX_STOCKS`, then run this section in Colab. The code downloads real stock or bond ticker data plus SPY, computes daily open-to-open returns, builds causal per-asset features, and packages everything into `RawExperimentData` for the walk-forward runner.

In [57]:
TICKERS = [
    'XLK', 'XLV', 'QQQ', "XLE", "GLD", 'MTUM'
]
MAX_STOCKS = len(TICKERS)  # set to 100 after pasting your full universe
START = "2015-01-01"
END = "2026-07-25"
CACHE_DIR = "data/cache"
BENCHMARK_TICKER = "SPY"
REBALANCE_FREQUENCY = "daily"  # "daily" or "weekly"

universe = UniverseConfig(
    tickers=TICKERS,
    max_stocks=MAX_STOCKS,
    include_cash=False,
    benchmark_ticker=BENCHMARK_TICKER,
)
market_config = MarketDataConfig(
    universe=universe,
    start=START,
    end=END,
    cache_dir=CACHE_DIR,
)
selected_tickers = universe.selected_tickers
selected_tickers

('XLK', 'XLV', 'QQQ', 'XLE', 'GLD', 'MTUM')

In [58]:
def _returns_wide(open_to_open_returns: pl.DataFrame, tickers: tuple[str, ...]) -> pl.DataFrame:
    wide = (
        open_to_open_returns
        .select(["decision_date", "ticker", "return"])
        .pivot(index="decision_date", on="ticker", values="return", aggregate_function="first")
        .sort("decision_date")
    )
    return wide.select(["decision_date", *tickers]).fill_null(0.0)


def _spy_returns(open_to_open_returns: pl.DataFrame) -> pl.DataFrame:
    return (
        open_to_open_returns
        .select(["decision_date", pl.col("return").alias("spy_return")])
        .sort("decision_date")
        .drop_nulls()
    )


def _filter_features_to_common_dates(features: FeatureBundle, returns: pl.DataFrame, spy_returns: pl.DataFrame) -> FeatureBundle:
    common_dates = (
        returns.select("decision_date")
        .join(spy_returns.select("decision_date"), on="decision_date", how="inner")
        .rename({"decision_date": "date"})
        .with_columns(pl.col("date").cast(pl.Date))
        .unique()
        .sort("date")
    )
    asset = features.asset_features.join(common_dates, on="date", how="inner").sort(["date", "ticker"])
    macro = (
        common_dates
        .join(features.macro_features, on="date", how="left")
        .sort("date")
        .with_columns(pl.all().exclude("date").forward_fill().fill_null(0.0))
    )
    spectral = features.spectral_features.join(common_dates, on="date", how="inner").sort("date")
    dates = tuple(common_dates.get_column("date").to_list())
    return FeatureBundle(
        asset_features=asset,
        macro_features=macro,
        spectral_features=spectral,
        decision_dates=dates,
        tickers=features.tickers,
        asset_feature_columns=features.asset_feature_columns,
        macro_feature_columns=features.macro_feature_columns,
        spectral_feature_columns=features.spectral_feature_columns,
    )


def make_real_yfinance_data() -> RawExperimentData:
    ohlcv = download_ohlcv(selected_tickers, START, END, market_config)
    spy_ohlcv = download_ohlcv((BENCHMARK_TICKER,), START, END, market_config)
    macro = download_macro_series(START, END, market_config)
    calendar = build_rebalance_calendar(ohlcv, REBALANCE_FREQUENCY)

    market_bundle = MarketDataBundle(
        ohlcv=ohlcv,
        spy_ohlcv=spy_ohlcv,
        macro=macro,
        calendar=calendar,
    )
    features = build_feature_bundle(
        market_bundle,
        FeatureConfig(
            accumulation_window=40,
            klinger_fast_span=34,
            klinger_slow_span=55,
            klinger_signal_span=13,
            macd_fast_span=12,
            macd_slow_span=26,
            macd_signal_span=9,
            mr_ewma_span=200,
            mr_vol_window=200,
            spectral_dim=20,
            cmf_window=60,
        ),
    )

    stock_returns = _returns_wide(
        compute_open_to_open_returns(ohlcv, calendar),
        selected_tickers,
    )
    spy_returns = _spy_returns(compute_open_to_open_returns(spy_ohlcv, calendar))
    features = _filter_features_to_common_dates(features, stock_returns, spy_returns)
    common_dates = pl.DataFrame({"decision_date": list(features.decision_dates)}).with_columns(pl.col("decision_date").cast(pl.Date))
    stock_returns = common_dates.join(stock_returns, on="decision_date", how="inner")
    spy_returns = common_dates.join(spy_returns, on="decision_date", how="inner")
    return RawExperimentData(features=features, returns=stock_returns, spy_returns=spy_returns)


raw_data = make_real_yfinance_data()
raw_data.features.asset_features.tail(), raw_data.returns.tail(), raw_data.spy_returns.tail()

(shape: (5, 16)
 ┌────────────┬────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ date       ┆ ticker ┆ mr_ewma50_ ┆ ewma50_sl ┆ … ┆ cmf_days_ ┆ frog_in_t ┆ bollinger ┆ fip_over_ │
 │ ---        ┆ ---    ┆ vol_gap    ┆ ope       ┆   ┆ since_cro ┆ he_pan    ┆ _bandwidt ┆ bollinger │
 │ date       ┆ str    ┆ ---        ┆ ---       ┆   ┆ ss        ┆ ---       ┆ h         ┆ _bandwidt │
 │            ┆        ┆ f64        ┆ f64       ┆   ┆ ---       ┆ f64       ┆ ---       ┆ h         │
 │            ┆        ┆            ┆           ┆   ┆ i64       ┆           ┆ f64       ┆ ---       │
 │            ┆        ┆            ┆           ┆   ┆           ┆           ┆           ┆ f64       │
 ╞════════════╪════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 2026-07-09 ┆ MTUM   ┆ -9.115294  ┆ 5.614048  ┆ … ┆ 65        ┆ 1.264911  ┆ 0.129211  ┆ 9.789499  │
 │ 2026-07-09 ┆ QQQ    ┆ -9.139041  ┆ 5.44186   ┆ … ┆ 59        ┆ 

In [ ]:
n_stocks = len(raw_data.features.tickers)
n_tradable_assets = n_stocks + 1  # risky assets plus cash
asset_feature_dim = len(raw_data.features.asset_feature_columns)
macro_feature_dim = len(raw_data.features.macro_feature_columns)
routing = selected_direct_allocation_indices(raw_data.features.asset_feature_columns)

# DPO uses one complete chronological scan per epoch; there is no batch-size setting.
# It must execute the same dense allocations used by its training objective.
# Top-N execution is available only for the environment-only path.
# Turnover and concentration are diagnostics; only lambda_drawdown affects DPO loss.
POLICY_MODE = "dpo"  # "dpo" or "equal_weight"
TOP_N_POSITIONS = None

config = ExperimentConfig(
    walk_forward=WalkForwardConfig(train_years=3, test_years=1, step_years=1),
    preprocessing=PreprocessingConfig(rolling_window=252),
    dpo=DPOConfig(
        learning_rate=1e-4,
        num_epochs=50,
        transaction_cost_bps=0.0,
        lambda_drawdown=0.03,
        allocation_hidden_dims=(16, 8),
        allocation_hidden_activation="tanh",
        allocation_output_activation="identity",
        allocation_use_layer_norm=True,
        simplex_activation="sparsemax",
    ),
    env=EnvConfig(
        sortino_target_return=0.0,
        sortino_downside_penalty=0.0,
        top_n_positions=TOP_N_POSITIONS,
        transaction_cost_rate=0.0
    ),
    enable_dpo=POLICY_MODE == "dpo",
    rebalance_frequency=REBALANCE_FREQUENCY,
    seed=6,
)

print({
    "stocks": n_stocks,
    "tradable_assets": n_tradable_assets,
    "asset_feature_dim": asset_feature_dim,
    "macro_feature_dim": macro_feature_dim,
    "policy_mode": POLICY_MODE,
    "top_n_positions": TOP_N_POSITIONS,
    "direct_features": len(routing.direct_allocation_indices),
    "decision_dates": len(raw_data.features.decision_dates),
})

result = run_walk_forward_experiment(raw_data, config)
metrics_to_frame(result)

{'stocks': 6, 'tradable_assets': 7, 'asset_feature_dim': 14, 'macro_feature_dim': 24, 'policy_mode': 'dpo', 'top_n_positions': None, 'direct_features': 14, 'decision_dates': 2895}


split_index,test_start,test_end,portfolio_cumulative_return,spy_cumulative_return,spy_relative_alpha,portfolio_max_drawdown,portfolio_mean_turnover,portfolio_total_transaction_cost,portfolio_sharpe_ratio,portfolio_sortino_ratio,portfolio_calmar_ratio,tracking_error,information_ratio,beta,regression_alpha
i64,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,2018-01-01,2018-12-31,0.094793,-0.059544,0.154337,0.081949,0.325233,0.0,0.813818,1.180025,1.161561,0.160573,0.901916,0.312609,0.112795
1,2019-01-01,2019-12-31,0.328251,0.3183,0.009951,0.098118,0.272521,0.0,2.55706,4.310136,3.345465,0.077525,0.096084,0.764439,0.074114
2,2020-01-01,2020-12-31,0.335411,0.167561,0.16785,0.174686,0.33092,0.0,1.161701,1.721656,1.911345,0.211735,0.613828,0.701941,0.189034
3,2021-01-01,2021-12-31,0.407963,0.319228,0.088735,0.080804,0.33497,0.0,2.326127,3.633943,5.048805,0.109303,0.635187,0.883585,0.102549
4,2022-01-01,2022-12-31,-0.015284,-0.187437,0.172153,0.24598,0.381663,0.0,0.047927,0.066107,-0.062381,0.124421,1.529819,0.822619,0.158533
5,2023-01-01,2023-12-31,0.347676,0.246359,0.101317,0.086254,0.297825,0.0,1.837237,2.874908,4.068197,0.12014,0.70376,0.916723,0.103798
6,2024-01-01,2024-12-31,0.254306,0.264941,-0.010636,0.08571,0.388068,0.0,1.644796,2.312497,2.967058,0.101964,-0.058648,0.828032,0.035823
7,2025-01-01,2025-12-31,0.128306,0.182241,-0.053935,0.153677,0.3137,0.0,0.713936,1.1023,0.841996,0.111255,-0.435184,0.821308,-0.014559
8,2026-01-01,2026-12-31,0.170891,0.101242,0.069649,0.099648,0.395412,0.0,1.496413,2.353594,3.655277,0.196302,0.698971,0.837349,0.16953


## Performance vs S&P 500


In [60]:
performance_fig = build_performance_figure(result)
performance_fig.show()


## Drawdown

Compare portfolio and SPY peak-to-trough declines over the walk-forward period.


In [61]:
drawdown_fig = build_drawdown_figure(result)
drawdown_fig.show()


## Portfolio Allocation


In [62]:
allocation_fig = build_allocation_figure(result)
allocation_fig.show()


## Holdings Heatmap


In [63]:
holdings_heatmap_fig = build_holdings_heatmap_granular(
    result,
    min_weight=0.001,
    top_n=50,
    freq=None,
    include_cash=False,
)
holdings_heatmap_fig.show()


## Regime Portfolio (Deferred)

Macro and regime features are intentionally excluded from the current
research path.


In [64]:
print("Macro/regime features are deferred; no regime model is run.")


Macro/regime features are deferred; no regime model is run.


In [65]:
# Reproducible artifact export and release validation
import numpy as np
from finrl.experiments import build_run_metadata, save_walk_forward_artifacts
from finrl.backtest.release import validate_release
metadata = build_run_metadata(raw_data, CONFIG)
save_walk_forward_artifacts(result, metadata, "walk_forward_artifacts")
print(f"Run ID: {metadata.run_id}")
print("Use finrl.backtest.robustness.stress_transaction_costs and")
print("validate_release(...) for cost, delay, capacity, and gate checks.")

NameError: name 'CONFIG' is not defined